In [3]:
# Load all supervised ML models
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier, VotingRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, AdaBoostClassifier, AdaBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.decomposition import PCA
from sklearn.datasets import *
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
import os
import ai
from sklearn.model_selection import cross_val_predict
import warnings
warnings.filterwarnings('ignore')
print('All Modules Loaded Successfully11!!')

All Modules Loaded Successfully11!!


In [ ]:
# !pip install streamlit pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost

In [ ]:
sklearn_datasets = [load_breast_cancer, load_diabetes,load_digits, load_iris, load_wine,fetch_california_housing]



def load_dataset(dataset_name_no:"str,[index]" )->"Dataframe":
  '''This is Docstring'''
  dataset = dataset_name_no()
  df = pd.DataFrame(dataset['data'], columns=dataset['feature_names'])
  df['target'] = pd.Series(dataset['target'])
  return {'dataframe':df, 'dataset_meta_info':dataset}

In [ ]:
print(load_dataset(sklearn_datasets[0])['dataset_meta_info']['DESCR'])

In [ ]:
df = load_dataset(sklearn_datasets[0])['dataframe']

In [ ]:
def identify_nan_values_columns(df:pd.DataFrame, threshold = 0.5)->"List":
  df_col_dict = dict(df.isna().mean().items())
  func = lambda data: data[0]

  df_col_dict = dict(df.isna().mean().items())

  cols = list(map(func, filter(lambda item: item[1] > threshold, df_col_dict.items())))

  nan_cols = cols
  return nan_cols

In [ ]:
# fill_na(adf['Salary'], 'mean')
def fill_na(columns: pd.Series, type: str = 'median'):

    if pd.api.types.is_numeric_dtype(columns):
        if type == 'mean':
            return columns.fillna(columns.mean())
        elif type == 'median':
            return columns.fillna(columns.median())
        else:
            return columns.fillna(0)

    elif pd.api.types.is_datetime64_any_dtype(columns):
        if type == 'mean':
            return columns.fillna(columns.mean())
        elif type == 'median':
            return columns.fillna(columns.median())
        else:
            # fallback: forward fill (more practical for time data)
            return columns.fillna(method='ffill')

    elif pd.api.types.is_object_dtype(columns):
        return columns.fillna(columns.mode()[0])

    return columns

In [ ]:
def fix_fill_na(df,thresh):
  for i in identify_nan_values_columns(df,thresh):
    df[i] = fill_na(df[i], 'mean')

In [ ]:
def drop_cols(df:pd.DataFrame, cols:list)->pd.DataFrame:
  return df.drop(cols, axis=1)

In [1]:
def textual_to_numerical(df, method = 'le'):
  tdf = df.copy()
  textual_df = tdf.select_dtypes('object')

  if method == 'le':
    le = LabelEncoder()
    for col in textual_df.columns:
      tdf[col] = le.fit_transform(tdf[col])

  elif method == 'get_dummies':
    tdf = pd.concat([tdf, pd.get_dummies(textual_df, dtype=int, drop_first=True)], axis=1)
    tdf.drop(textual_df.columns, axis=1, inplace=True)

  return tdf



In [2]:
def change_textual_todate(df):
  date_col = list(filter(lambda col: 'date' in col.lower(), df.columns))
  if len(date_col) == 0:
    return df
  df[date_col[0]] = pd.to_datetime(df[date_col[0]], format='mixed')
  return df

In [ ]:
def change_data_scale(df, method:'str' = 'min_max'):
  date_col = list(filter(lambda col: 'date' in col.lower(), df.columns))
  yes_date = False
  try:
    if len(date_col)>0:
      tdf = df.drop(date_col, axis=1)
      yes_date = True
    else:
      raise 'Date not found'
  except:
    tdf = df.copy()

  if method == 'min_max':
    scaler = MinMaxScaler()
    tdf = pd.DataFrame(scaler.fit_transform(tdf), columns=tdf.columns)
  elif method == 'ss':
    scaler = StandardScaler()
    tdf = pd.DataFrame(scaler.fit_transform(tdf), columns=tdf.columns)

  if yes_date:
    print('sdfgbn')
    tdf[date_col[0]] = df[date_col[0]]
  return tdf



In [ ]:
def return_target(df, target = None):
  return df[target]

In [ ]:
def cross_model(base_model,X,y,cv):
  score = cross_val_score(base_model, X, y, cv = cv).mean()
  return score

In [ ]:
def choose_problem_type(type_='classification', problem = None):

  all_reg_model = [LinearRegression, SVR, KNeighborsRegressor, DecisionTreeRegressor, RandomForestRegressor,
                 GradientBoostingRegressor, AdaBoostRegressor, LGBMRegressor, XGBRegressor]

  all_class_model = [LogisticRegression, 
                     #SVC, 
                     KNeighborsClassifier, DecisionTreeClassifier, RandomForestClassifier,
                   GradientBoostingClassifier, AdaBoostClassifier, LGBMClassifier, XGBClassifier, GaussianNB, BernoulliNB, MultinomialNB]




  if type_ == 'classification':
    global selected_model
    selected_model = all_class_model

  elif type_ == 'regression':
    selected_model = all_reg_model


  for i in selected_model:
    try:
      if i.__name__[:4] == 'LGBM':
        if problem == 'timeseries':
          model = i()
          model.fit(X,y)
          model_result = model.score(X,y)
        else:
          model_result = cross_model(i(verbose = -1),X,y,5)

      else:
        if problem == 'timeseries':
          model = i()
          model.fit(X,y)
          model_result = model.score(X,y)
        else:
          model_result = cross_model(i(),X,y,5)
      print(f'Model Name: {i.__name__} | Score: {model_result}')
    except:
      print('Something went wrong')

In [ ]:
def cross_model_predict(base_model,X,y,cv):
  y_pred = cross_val_predict(base_model, X, y, cv = cv)
  return y_pred

In [ ]:
def run_ml_models_and_predict_output(tempData, problem_type = 'regression'):
  # tempData = X.iloc[5,:].to_numpy().reshape(1,-1)
  all_model_class_pred = {'Model_Name':[], 'Model_output':[], 'Predict_prob':[]}

  for i in selected_model:
    try:
      if i.__name__[:4] == 'LGBM':
        model = model_predict(i(num_leaves = 100, max_depth=2,min_child_samples=5, verbose = -1),X,y)
      else:
        model = model_predict(i(),X,y)


      final_ans = model.predict(tempData)[0]
      if problem_type == 'classification':
        prob = model.predict_proba(tempData).max()
      prob = 1

      print(f'Model Name: {i.__name__} | Prediction: {final_ans}')

      all_model_class_pred['Model_Name'].append(i.__name__)
      all_model_class_pred['Model_output'].append(final_ans)
      all_model_class_pred['Predict_prob'].append(prob)
    except:
      pass


  return all_model_class_pred

In [ ]:
def model_predict(model,X_data, y_data):
  model = model
  model.fit(X_data, y_data)
  return model

In [ ]:
def final_prediction_with_prob(dataframe, type = 'regression'):
  model_compare_df = dataframe
  if type == 'regression':
    return (model_compare_df['Model_output'].mean(), model_compare_df['Predict_prob'].mean())
  else:
    return (model_compare_df['Model_output'].mode().values[0], model_compare_df['Predict_prob'].mean())

In [ ]:
# !pip install kagglehub

In [ ]:
# import kagglehub

# kaggle_url = input('Enter url: ')
# # Download latest version
# if '?' not in kaggle_url.split('datasets')[-1][1:]:
#   sub_url = kaggle_url.split('datasets')[-1][1:]+'?'

# path = kagglehub.dataset_download(f"{sub_url[:sub_url.find('?')]}")
# file_name = os.listdir(path)[0]
# file_path = path + '/' + file_name
# file_path

In [ ]:
file_path = r"C:\Users\salma\Downloads\archive (16)\Paisabazaar Banking Fraud Dataset.csv"

df = ai.load_dataset(file_path)
df

In [ ]:
ai.show_textual_analysis(df)

In [ ]:
ai.mannual_summary(df)

In [ ]:
ai.show_numerical_analysis(df)

In [ ]:
ai.show_textual_analysis(df)

In [ ]:
features_col = []
target_col = df.columns[-1]
drop_col =  ['ID', 'Customer_ID', 'Month', 'Name', 'SSN', 'Delay_from_due_date']

In [ ]:
df = drop_cols(df,drop_col)
df

In [ ]:
identify_nan_values_columns(df,0.05)

In [ ]:
fix_fill_na(df,0.05)

In [ ]:
df = textual_to_numerical(df)
df.sample(3)

In [ ]:
df = change_textual_todate(df)
df.sample(3)

In [ ]:
target = target_col
X = df.drop(target, axis = 1)
y = return_target(df, target)


In [ ]:
X.sample(5)

In [ ]:
y.sample(5)

In [ ]:
X = change_data_scale(X, method = 'ss')
X.sample(5)

In [ ]:
choose_problem_type()

In [ ]:
all_model_class_pred =  run_ml_models_and_predict_output(X.iloc[7,:].to_numpy().reshape(1,-1))

model_compare_df = pd.DataFrame(all_model_class_pred)

In [ ]:
final_prediction_with_prob(model_compare_df)